# UdaciSense: Multi-Stage Compression Pipeline - Google Colab Pro

**🎯 Final Integration: Sequential Distillation → Quantization Pipeline**

**Strategy**: Combine the best individual techniques in a sequential pipeline to achieve all CTO targets:
- 70% model size reduction (5.96 MB → 1.79 MB)
- 60% inference speedup (10.56 ms → 4.22 ms) 
- <5% accuracy drop (maintain >83% from 87.40% baseline)

**Expected Result**: 80%+ size reduction, maintaining ~82% accuracy

In [ ]:
# Cell 1: Mount Google Drive and Setup
from google.colab import drive
import os
import sys
import warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')

# UPDATE THIS PATH to your Google Drive project location
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/udacity-ml-compression-pipeline/project/starter_kit'
os.chdir(DRIVE_PROJECT_PATH)
print(f"✅ Changed to directory: {os.getcwd()}")

# Add to Python path
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)
src_dir = os.path.join(current_dir, 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

In [ ]:
# Cell 2: Install packages with UV (faster)
!curl -LsSf https://astral.sh/uv/install.sh | sh
!/root/.local/bin/uv pip install --system torch>=2.0.0 torchvision>=0.15.0
!/root/.local/bin/uv pip install --system matplotlib seaborn pandas scikit-learn pillow tqdm plotly
!/root/.local/bin/uv pip install --system thop

print("✅ All packages installed with UV!")

In [ ]:
# Cell 3: GPU Setup
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cpu_device = torch.device('cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🚀 GPU Available: {gpu_name}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
    torch.cuda.empty_cache()
else:
    print("⚠️ No GPU found")

print(f"Device: {device}")

In [ ]:
# Cell 4: Import Project Modules
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# Import project modules
from src.utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP, TARGET_MODEL_COMPRESSION
from src.utils.data_loader import get_household_loaders
from src.utils.model import load_model, save_model
from src.utils.compression import evaluate_optimized_model, compare_optimized_model_to_baseline
from src.utils.evaluation import evaluate_model_metrics
from src.compression.in_training.distillation import train_with_distillation, MobileNetV3_Household_Small

print("✅ All modules imported successfully")
print(f"🎯 Targets: {TARGET_MODEL_COMPRESSION*100}% size reduction, {TARGET_INFERENCE_SPEEDUP*100}% speedup, <{MAX_ALLOWED_ACCURACY_DROP*100}% accuracy drop")

In [ ]:
# Cell 5: Load Dataset
train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", 
    batch_size=256, 
    num_workers=2
)
class_names = train_loader.dataset.classes
input_size = (1, 3, 32, 32)

print(f"✅ Dataset loaded: {len(class_names)} classes")
print(f"Train samples: {len(train_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")

In [ ]:
# Cell 6: Load Baseline Model and Metrics
print("📊 Loading baseline model and metrics...")

# Load baseline
baseline_model_path = "models/baseline_mobilenet_colab/checkpoints/model.pth"
baseline_metrics_path = "results/baseline_mobilenet_colab/metrics.json"

baseline_model = load_model(baseline_model_path, device)
with open(baseline_metrics_path, 'r') as f:
    baseline_metrics = json.load(f)

# Calculate targets
target_size_mb = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
target_cpu_time = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
min_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP)

print(f"📋 BASELINE PERFORMANCE:")
print(f"   Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   CPU Time: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")

print(f"\n🎯 CTO TARGETS:")
print(f"   Target Size: ≤{target_size_mb:.2f} MB")
print(f"   Target Speed: ≤{target_cpu_time:.2f} ms")
print(f"   Target Accuracy: ≥{min_accuracy:.2f}%")

In [ ]:
# Cell 7: FINAL Fixed Pipeline Implementation Class
class SequentialOptimizationPipeline:
    """Multi-stage compression pipeline implementing Sequential Distillation → Quantization"""
    
    def __init__(self, baseline_metrics):
        self.baseline_metrics = baseline_metrics
        self.results_history = []
        
    def create_pipeline_directories(self):
        """Create directories for pipeline results"""
        pipeline_dirs = [
            "models/pipeline/stage1_distillation",
            "models/pipeline/final_distilled_quantized", 
            "results/pipeline/stage1_distillation",
            "results/pipeline/final_distilled_quantized"
        ]
        
        for dir_path in pipeline_dirs:
            os.makedirs(dir_path, exist_ok=True)
        print("📁 Pipeline directories created")
    
    def stage1_knowledge_distillation(self, teacher_model):
        """Stage 1: Knowledge Distillation"""
        print("\n🔄 STAGE 1: Knowledge Distillation")
        print("=" * 50)
        
        # Create smaller student model and move to GPU
        student_model = MobileNetV3_Household_Small(num_classes=len(class_names))
        student_model = student_model.to(device)
        
        # Create actual optimizer and criterion objects
        optimizer = optim.Adam(student_model.parameters(), lr=0.001, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
        
        # FINAL FIX: Complete training configuration with patience parameter
        training_config = {
            'temperature': 4.0,
            'alpha': 0.7,
            'num_epochs': 20,
            'optimizer': optimizer,
            'criterion': criterion,
            'scheduler': scheduler,
            'patience': 10,               # ADDED: early stopping patience
            'grad_clip_norm': 1.0,
            'device': device
        }
        
        # Train with distillation
        student_model, training_stats, best_accuracy, best_epoch = train_with_distillation(
            student_model=student_model,
            teacher_model=teacher_model,
            train_loader=train_loader,
            test_loader=test_loader,
            training_config=training_config,
            checkpoint_path="models/pipeline/stage1_distillation/model.pth"
        )
        
        # Evaluate Stage 1
        stage1_results = self.evaluate_pipeline_stage(
            student_model, 
            "stage1_distillation", 
            reference_model=teacher_model,
            device_eval=device
        )
        
        self.results_history.append(('Stage 1: Distillation', stage1_results))
        return student_model, stage1_results
    
    def stage2_quantization(self, input_model):
        """Stage 2: Dynamic Quantization"""
        print("\n🔄 STAGE 2: Dynamic Quantization")
        print("=" * 50)
        
        # Apply quantization
        quantized_model = self.apply_dynamic_quantization(input_model, "final_pipeline")
        
        # Save quantized model
        save_path = "models/pipeline/final_distilled_quantized/model.pth"
        torch.save(quantized_model.state_dict(), save_path)
        print(f"💾 Saved final pipeline model to {save_path}")
        
        # Evaluate Stage 2
        stage2_results = self.evaluate_pipeline_stage(
            quantized_model, 
            "final_distilled_quantized", 
            reference_model=baseline_model,
            device_eval=cpu_device
        )
        
        self.results_history.append(('Stage 2: Quantization', stage2_results))
        return quantized_model, stage2_results
    
    def apply_dynamic_quantization(self, model, stage_name="quantization"):
        """Apply dynamic quantization to the input model"""
        print(f"🔧 Applying dynamic quantization for {stage_name}...")
        
        model_cpu = model.to(cpu_device)
        model_cpu.eval()
        
        quantized_model = torch.quantization.quantize_dynamic(
            model_cpu,
            {torch.nn.Linear, torch.nn.Conv2d},
            dtype=torch.qint8
        )
        
        print(f"✅ Dynamic quantization applied")
        return quantized_model
    
    def evaluate_pipeline_stage(self, model, stage_name, reference_model=None, device_eval=None):
        """Evaluate a pipeline stage and save results"""
        if device_eval is None:
            device_eval = cpu_device
            
        print(f"📊 Evaluating {stage_name}...")
        
        experiment_name = f"pipeline/{stage_name}"
        
        stage_results = evaluate_optimized_model(
            model, 
            test_loader, 
            experiment_name, 
            class_names, 
            input_size,
            device=device_eval
        )
        
        return stage_results
    
    def run_complete_pipeline(self, teacher_model):
        """Run the complete sequential pipeline"""
        print("🚀 STARTING SEQUENTIAL DISTILLATION → QUANTIZATION PIPELINE")
        print("=" * 70)
        
        self.create_pipeline_directories()
        
        # Stage 1: Knowledge Distillation
        distilled_model, stage1_results = self.stage1_knowledge_distillation(teacher_model)
        
        # Stage 2: Quantization
        final_model, stage2_results = self.stage2_quantization(distilled_model)
        
        # Final summary
        self.print_pipeline_summary()
        
        return final_model, self.results_history
    
    def print_pipeline_summary(self):
        """Print comprehensive pipeline results summary"""
        print("\n" + "=" * 70)
        print("📊 COMPLETE PIPELINE RESULTS SUMMARY")
        print("=" * 70)
        
        baseline_size = self.baseline_metrics['size']['model_size_mb']
        baseline_acc = self.baseline_metrics['accuracy']['top1_acc']
        baseline_time = self.baseline_metrics['timing']['cpu']['avg_time_ms']
        
        print(f"📋 BASELINE: {baseline_acc:.2f}% acc, {baseline_size:.2f} MB, {baseline_time:.2f} ms")
        
        for stage_name, results in self.results_history:
            acc = results['accuracy']['top1_acc']
            size = results['size']['model_size_mb']
            cpu_time = results['timing']['cpu']['avg_time_ms']
            
            size_reduction = (1 - size/baseline_size) * 100
            acc_drop = baseline_acc - acc
            speed_improvement = (1 - cpu_time/baseline_time) * 100 if cpu_time < baseline_time else -(cpu_time/baseline_time - 1) * 100
            
            print(f"📋 {stage_name}: {acc:.2f}% acc ({acc_drop:+.2f}%), {size:.2f} MB ({size_reduction:.1f}% ↓), {cpu_time:.2f} ms ({speed_improvement:+.1f}%)")
        
        # Check final targets
        if self.results_history:
            final_results = self.results_history[-1][1]
            final_size = final_results['size']['model_size_mb']
            final_acc = final_results['accuracy']['top1_acc']
            final_time = final_results['timing']['cpu']['avg_time_ms']
            
            target_size = baseline_size * (1 - TARGET_MODEL_COMPRESSION)
            target_acc = baseline_acc * (1 - MAX_ALLOWED_ACCURACY_DROP)
            target_time = baseline_time * (1 - TARGET_INFERENCE_SPEEDUP)
            
            print(f"\n🎯 TARGET ACHIEVEMENT:")
            print(f"   Size: {final_size:.2f} MB ≤ {target_size:.2f} MB? {'✅' if final_size <= target_size else '❌'}")
            print(f"   Accuracy: {final_acc:.2f}% ≥ {target_acc:.2f}%? {'✅' if final_acc >= target_acc else '❌'}")
            print(f"   Speed: {final_time:.2f} ms ≤ {target_time:.2f} ms? {'✅' if final_time <= target_time else '❌'}")

print("✅ FINAL Fixed Pipeline class defined")

In [ ]:
# Cell 8: Execute Complete Pipeline
print("🚀 Initializing and running complete optimization pipeline...")

# Initialize pipeline
pipeline = SequentialOptimizationPipeline(baseline_metrics)

# Run complete pipeline
final_optimized_model, pipeline_results = pipeline.run_complete_pipeline(baseline_model)

print("\n🎉 Pipeline execution completed!")

In [ ]:
# Cell 9: Final Analysis and Visualization
print("📊 Generating final analysis and visualizations...")

# Create comparison DataFrame
comparison_data = []

# Add baseline
comparison_data.append({
    'Model': 'Baseline',
    'Accuracy (%)': baseline_metrics['accuracy']['top1_acc'],
    'Size (MB)': baseline_metrics['size']['model_size_mb'],
    'CPU Time (ms)': baseline_metrics['timing']['cpu']['avg_time_ms'],
    'Size Reduction (%)': 0.0,
    'Accuracy Drop (%)': 0.0
})

# Add pipeline stages
baseline_size = baseline_metrics['size']['model_size_mb']
baseline_acc = baseline_metrics['accuracy']['top1_acc']

for stage_name, results in pipeline_results:
    size_reduction = (1 - results['size']['model_size_mb']/baseline_size) * 100
    acc_drop = baseline_acc - results['accuracy']['top1_acc']
    
    comparison_data.append({
        'Model': stage_name,
        'Accuracy (%)': results['accuracy']['top1_acc'],
        'Size (MB)': results['size']['model_size_mb'],
        'CPU Time (ms)': results['timing']['cpu']['avg_time_ms'],
        'Size Reduction (%)': size_reduction,
        'Accuracy Drop (%)': acc_drop
    })

# Create DataFrame and display
df_comparison = pd.DataFrame(comparison_data)
print("\n📊 COMPLETE PIPELINE COMPARISON:")
print(df_comparison.round(2))

# Save comparison results
df_comparison.to_csv('results/pipeline_comparison.csv', index=False)
print("\n💾 Results saved to results/pipeline_comparison.csv")

print("\n🎉 PIPELINE ANALYSIS COMPLETE!")